In [ ]:
import pandas as pd
import numpy as np
import duckdb
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder

# Set token
HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("Please set HF_TOKEN environment variable")

# Connect to DuckDB
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

# Load data from Hugging Face
MONTH = "2026-03"
df = con.sql(f"""
    SELECT 
        content_hash_id,
        client_hash_id,
        gsc_impressions,
        gsc_clicks,
        gsc_sum_position,
        scroll_events,
        month,
        sessions_ai,
        ai_chatgpt,
        ai_perplexity,
        ai_gemini,
        ai_copilot,
        ai_claude,
        ai_meta,
        ai_other
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={MONTH}/*.parquet')
""").df()

print(f"Loaded {len(df)} rows")

# --- FIX: Convert month from '2026-03' to number (3) ---
df['month'] = df['month'].str.split('-').str[1].astype(int)

# Clean data - remove rows with missing values
df = df.dropna()
print(f"After cleaning: {len(df)} rows")

# Encode categorical columns
le_content = LabelEncoder()
le_client = LabelEncoder()
df['content_encoded'] = le_content.fit_transform(df['content_hash_id'].astype(str))
df['client_encoded'] = le_client.fit_transform(df['client_hash_id'].astype(str))

# Select features (all numeric now)
features = [
    'gsc_impressions',
    'gsc_clicks',
    'gsc_sum_position',
    'scroll_events',
    'month',  # Now numeric!
    'sessions_ai',
    'ai_chatgpt',
    'ai_perplexity',
    'ai_gemini',
    'ai_copilot',
    'ai_claude',
    'ai_meta',
    'ai_other',
    'content_encoded',
    'client_encoded'
]

X = df[features]
y = df['gsc_impressions']

print(f"Features: {len(features)}")
print(f"Target: gsc_impressions")

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Train: {len(X_train)} rows")
print(f"Test: {len(X_test)} rows")

# Train Random Forest
model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

# Predictions
train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

# Metrics
train_rmse = np.sqrt(mean_squared_error(y_train, train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, test_pred))
test_mae = mean_absolute_error(y_test, test_pred)
test_r2 = r2_score(y_test, test_pred)

print("\n" + "=" * 60)
print("MODEL PERFORMANCE")
print("=" * 60)
print(f"Train RMSE: {train_rmse:.2f}")
print(f"Test RMSE: {test_rmse:.2f}")
print(f"Test MAE: {test_mae:.2f}")
print(f"Test R²: {test_r2:.3f}")

# Baseline comparison
baseline_rmse = 1200
improvement = ((baseline_rmse - test_rmse) / baseline_rmse) * 100

print("\n" + "=" * 60)
print("BASELINE COMPARISON")
print("=" * 60)
print(f"Baseline RMSE: {baseline_rmse:.2f}")
print(f"Model RMSE: {test_rmse:.2f}")
print(f"Improvement: {improvement:.1f}%")

if test_rmse < baseline_rmse:
    print("✅ Model BEATS the baseline!")
else:
    print("⚠️ Model does NOT beat the baseline. Try tuning.")

# Feature Importance
print("\n" + "=" * 60)
print("FEATURE IMPORTANCE")
print("=" * 60)
importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print(importance)

con.close()